In [ ]:
import MetaTrader5 as mt5

from src.config import CONFIG
from src.loaders import load_mt5_live, fetch_mt5_history
from src.signals import SignalResult, generate_signal, regime_gate, apply_filters
from src.live_runner import run_live
from src.exports import save_artefacts
from src.mql5_overlay import write_mql5_overlay

### Section 13 — Live Mode (MT5 Adapter)

Live mode uses the same core probability band engine as the backtest workflow. The main difference is the data source — MT5 live bars instead of historical CSV data.

This section defines the MT5-connected live runner and prepares the artefacts required for live monitoring, signal updates, and external overlays.

The live notebook is designed to reuse the shared `src/` modules wherever possible, while keeping MT5-specific orchestration here.

In [ ]:
print("✅ Live mode functions loaded")
print(f"   Instrument : {CONFIG['instrument']}")
print(f"   Timeframe  : {CONFIG['timeframe']}")
print("   run_live() available")
print("   save_artifacts() available")
print("   fetch_mt5_history() available")

In [ ]:
# Optional MT5 historical backfill for local reuse
# import MetaTrader5 as mt5
#
# df_hist = fetch_mt5_history(
#     symbol=CONFIG['instrument'],
#     timeframe_mt5=mt5.TIMEFRAME_M1,
#     lookback_days=30,
#     save_csv=True,
#     csv_dir="data",
# )
#
# df_hist.head(3)

---
### Section 17 — MQL5 Overlay Generator

This section writes the MQL5 indicator source file to disk.
The indicator reads `live_artifacts/live_state.json` once per bar
and draws VWAP, sigma bands, zone label, and signal alert on the chart.

It contains no calculation logic — it is purely a rendering layer
that consumes the Python engine's JSON output.

**To install:**
1. Copy the generated `.mq5` file to your MT5 `Indicators/` folder
2. Compile in MetaEditor (F5)
3. Attach to the chart — set the JSON path to match your Python output path

In [ ]:
write_mql5_overlay(output_dir="live_artifacts/exports")

### Section 18 — Run Live Engine

In [ ]:
# LIVE MODE — run this cell last, after all cells above are done
# MT5 must be open and logged in before running this

CONFIG['instrument'] = 'US100.cash'

if 'prob_table' not in dir():
    print("❌ prob_table not found — load/export the probability table first")
    raise SystemExit

mt5.shutdown()
if not mt5.initialize():
    print(f"❌ MT5 initialize() failed — error: {mt5.last_error()}")
    print("   Make sure MT5 is open, logged in, and Algo Trading is enabled")
    print("   Tools → Options → Expert Advisors → tick 'Allow algorithmic trading'")
    raise SystemExit

print(f"✅ MT5 connected — {mt5.terminal_info().name}")
print(f"   Account : {mt5.account_info().login} | Server: {mt5.account_info().server}")

symbol_info = mt5.symbol_info(CONFIG['instrument'])

if symbol_info is None:
    print(f"\n❌ Symbol '{CONFIG['instrument']}' not found on this account.")
    print("   Symbols on this broker containing US/NAS/TECH:")
    all_syms = mt5.symbols_get()
    matches = [s.name for s in all_syms
               if any(x in s.name.upper() for x in ['US1', 'NAS', 'TECH', 'NDX', 'USTEC'])]
    print(f"   {matches[:20]}")
    print("\n   Update CONFIG['instrument'] to the correct name above, then re-run.")
    mt5.shutdown()
    raise SystemExit

live_spread = symbol_info.ask - symbol_info.bid
print(f"\n✅ Symbol found: {symbol_info.name}")
print(f"   Bid: {symbol_info.bid}  Ask: {symbol_info.ask}")
print(f"   Live spread right now: {live_spread:.1f} points")
print(f"   ↳ If this looks right, set CONFIG['spread_cost'] = {live_spread:.1f} and re-run relevant calibration cells")
print(f"   ↳ (only matters for outcome labelling accuracy, not for live signal generation)")

print("\n🟢 Starting live engine — press the ■ stop button in Jupyter to stop\n")
try:
    run_live(
        symbol=CONFIG['instrument'],
        timeframe_mt5=mt5.TIMEFRAME_M1,
        config=CONFIG,
        prob_table=prob_table,
        marginal_table=prob_table,
        load_mt5_live=load_mt5_live,
        EngineState=EngineState,
        update_engine_state=update_engine_state,
        generate_signal=generate_signal,
        regime_gate=regime_gate,
        apply_filters=apply_filters,
    )
finally:
    mt5.shutdown()
    print("\n🔴 MT5 connection closed cleanly")